In [1]:
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt, json 
from utils import plot_var_bias_without_u2
if not os.path.exists('help_figures'):
    os.makedirs('help_figures')

def plot_dctb(results_list,            # [results1, results3, results5]
              prop_weights_list,       # [1, 3, 5]
              S_t,                    # drei rote Referenzwerte
              xlim,                   # (xmin, xmax)
              label_style="plain",
              patient=""):   # "plain" oder "math"
    """
    Strip-Plot mit 3 Zeilen – nur DCTB.

    label_style:
        "plain" →  DCTB  (|{w_i=0}|=0.1)      ← robust
        "math"  →  DCTB  $|\{w_i=0\}|=0.1$    ← LaTeX-Look
    """

    # ---------- Label-Helfer --------------------------------------------------
    if label_style == "plain":
        row_label = lambda pw: f"(|{{w_i=0}}|={pw/10:.1f})"
    elif label_style == "math":
        row_label =  lambda pw: r"$p_{w_0} =$" + rf"{pw/10:.1f}"
    else:
        raise ValueError("label_style muss 'plain' oder 'math' sein.")

    # ---------- Daten zusammenführen -----------------------------------------
    frames = []
    for res, pw in zip(results_list, prop_weights_list):
        n = res["rf_pred"].shape[0]
        frames.append(pd.DataFrame({
            "value": res["rf_pred"],
            "row"  : [row_label(pw)] * n
        }))
    df = pd.concat(frames, ignore_index=True)

    row_order = [row_label(pw) for pw in prop_weights_list]   # 3 Zeilen

    # ---------- Plot ----------------------------------------------------------
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=(5.8, 2.6 + 0.35*len(row_order)))

    sns.stripplot(data=df, x="value", y="row",
                  order=row_order, jitter=0.25,
                  alpha=0.5, size=1.5, color="teal", ax=ax)

    # Mittelwerte
    for i, r in enumerate(row_order):
        m = df.loc[df["row"] == r, "value"].mean()
        ax.vlines(m, i-0.4, i+0.4, color="black", linestyle="--", linewidth=1)
        print(m)

    # Rote Referenz-Linien
    for pw, s in zip(prop_weights_list, S_t):
        i = row_order.index(row_label(pw))
        ax.vlines(s, i-0.4, i+0.4, color="red", linewidth=1)
        print(S_t)

    ax.set_ylabel("")
    if xlim is not None:
        ax.set_xlim(xlim)
    if patient == "averageS":
        ax.set_xlabel(r"$\hat{\theta}_{\tau}(x_{0})=\hat{P}\left(T^*>3 \mid X=x_{0}\right)$")
    elif patient == "lowerS":
        ax.set_xlabel(r"$\hat{\theta}_{\tau}(x_{low})=\hat{P}\left(T^*>3 \mid X=x_{low}\right)$")
    elif patient == "higherS":
        ax.set_xlabel(r"$\hat{\theta}_{\tau}(x_{high})=\hat{P}\left(T^*>3 \mid X=x_{high}\right)$")
    fig.tight_layout()

    # Optional speichern
    if not os.path.exists("help_figures"):
        os.makedirs("help_figures")
    fig.tight_layout()
    fig.savefig(f"help_figures/preds_dctb_{patient}.jpeg", dpi=300)
    plt.close()

def coverage_prop(ijk_std_estimate, S_dach, S):

    u = S_dach - 1.96 * ijk_std_estimate
    o = S_dach + 1.96 * ijk_std_estimate

    prop = np.sum((u <= S) & (o >= S)) / len(u)
    return prop.round(2)

def plot_dctb_variances_separate(results_list,        # [results1, results3, results5]
                                 prop_weights_list,   # [1, 3, 5]
                                 xlim2,               # tuple or dict[pw]->(xmin,xmax)
                                 patient=None,
                                 label_style="math"   # "plain" oder "math"
                                 ):
    """
    Erzeugt je einen separaten Plot für jede Zero-Weight-Stufe (|{w_i = 0}| / |{w_i}|),
    mit je 3 Zeilen: V̂_{IJ-U}^{wB}, V̂_{IJ-U}^{B}, V̂_{Boot}.

    Y-Achse zeigt nur die Namen der Schätzer, ohne Zusatz.
    Rote Linie = std(rf_pred) für die jeweilige Stufe.
    """

    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ---------- Label-Suffix -------------------------------------------------
    if label_style == "plain":
        suf = lambda pw: f" (|{{w_i=0}}|={pw/10:.1f})"
    elif label_style == "math":
        suf = lambda pw: rf"$\frac{{|\{{w_i = 0\}}|}}{{|\{{w_i\}}|}} = {pw/10:.1f}$"
    else:
        raise ValueError("label_style muss 'plain' oder 'math' sein.")

    base = {
        "wB":   r"$\hat{V}_{IJ-U}^{wB}$",
        "B":    r"$\hat{V}_{IJ-U}^{B}$",
        "boot": r"$\hat{V}_{Boot}$"
    }

    xlabel_map = {
        "averageS": r"$\hat{\sigma}\left(\hat{\theta}_{\tau}(x_{0})\right)$",
        "lowerS":   r"$\hat{\sigma}\left(\hat{\theta}_{\tau}(x_{low})\right)$",
        "higherS":  r"$\hat{\sigma}\left(\hat{\theta}_{\tau}(x_{high})\right)$"
    }

    frames, std_by_pw = [], {}

    # ---------- Daten sammeln ------------------------------------------------
    for res, pw in zip(results_list, prop_weights_list):
        v_wB   = res["ijk_u_jahn_var"].clip(lower=0).pipe(np.sqrt)
        v_B    = res["ijk_u_wager_var"].clip(lower=0).pipe(np.sqrt)
        v_boot = res["boot_var"].clip(lower=0).pipe(np.sqrt)

        std_rf = res["rf_pred"].std(ddof=1)
        std_by_pw[pw] = std_rf

        n = len(v_wB)
        frames.append(pd.DataFrame({
            "value": np.concatenate([v_wB, v_B, v_boot]),
            "row": [f"{base['wB']}  {suf(pw)}"] * n +
                   [f"{base['B']}   {suf(pw)}"] * n +
                   [f"{base['boot']} {suf(pw)}"] * n,
            "pw": [pw] * 3 * n
        }))

        print(f"ensoring scenario {pw} has emp. std.  {std_rf} .")
        print(f"Cencorsing scenario {pw} has max predicted std for weighted infinitesimal jackknife  {v_wB.max()} .")
        print(f"Cencorsing scenario {pw} has max predicted std for unweighted infinitesimal jackknife  {v_B.max()} .")
        print(f"Cencorsing scenario {pw} has max predicted std for bootstrap  {v_boot.max()} .")
        print(f"Cencorsing scenario {pw} has min predicted std for weighted infinitesimal jackknife  {v_wB.min()} .")
        print(f"Cencorsing scenario {pw} has min predicted std for unweighted infinitesimal jackknife  {v_B.min()} .")
        print(f"Cencorsing scenario {pw} has min predicted std for bootstrap  {v_boot.min()} .")
        print(f"Cencorsing scenario {pw} has mean predicted std for weighted infinitesimal jackknife  {v_wB.mean()} .")
        print(f"Cencorsing scenario {pw} has mean predicted std for unweighted infinitesimal jackknife  {v_B.mean()} .")
        print(f"Cencorsing scenario {pw} has mean predicted std for bootstrap  {v_boot.mean()} .")

    df_all = pd.concat(frames, ignore_index=True)

    # ---------- Einzelplots je Stufe -----------------------------------------
    sns.set_style("whitegrid")

    for pw in prop_weights_list:
        df_pw = df_all[df_all["pw"] == pw].copy()

        # Original-Labels mit w_i Zusatz
        full_labels = [
            f"{base['wB']}  {suf(pw)}",
            f"{base['B']}   {suf(pw)}",
            f"{base['boot']} {suf(pw)}"
        ]
        # Nur die Schätzer (für Anzeige)
        short_labels = [base["wB"], base["B"], base["boot"]]

        fig, ax = plt.subplots(figsize=(5.8, 3.0))

        sns.stripplot(data=df_pw, x="value", y="row",
                      order=full_labels, jitter=0.25,
                      alpha=0.5, size=1.5, color="teal", ax=ax)

        # Mittelwertlinien
        for i, r in enumerate(full_labels):
            m = df_pw.loc[df_pw["row"] == r, "value"].mean()
            ax.vlines(m, i - 0.4, i + 0.4, color="black",
                      linestyle="--", linewidth=1)

        # Rote Linie = std(rf_pred)
        std_val = std_by_pw[pw]
        ax.vlines(std_val, -0.4, 2.4, color="red", linewidth=1.2, zorder=10)

        ax.set_ylabel("")
        current_xlim = xlim2[pw] if isinstance(xlim2, dict) else xlim2
        if current_xlim is not None:
            ax.set_xlim(current_xlim)

        if patient in xlabel_map:
            ax.set_xlabel(xlabel_map[patient])

        # Nur kurze Labels anzeigen
        ax.set_yticks(range(len(short_labels)))
        ax.set_yticklabels(short_labels)

        fig.tight_layout()

        # Speichern
        if not os.path.exists("help_figures"):
            os.makedirs("help_figures")
        fig.tight_layout()
        fig.savefig(f"help_figures/dctb_variances_{patient}_pw{pw}.jpeg", dpi=300)
        plt.close()

def plot_dctb_all_variances_separate(results_list,        # [results1, results3, results5]
                                     prop_weights_list,   # [1, 3, 5]
                                     xlim2,               # tuple or dict[pw]->(xmin,xmax)
                                     patient=None,
                                     label_style="math"   # "plain" oder "math"
                                     ):
    """
    Erzeugt je einen separaten Plot für jede Zero-Weight-Stufe (|{w_i = 0}| / |{w_i}|),
    mit je 5 Zeilen: V̂_{IJ}^{wB}, V̂_{IJ-U}^{wB}, V̂_{IJ}^{B}, V̂_{IJ-U}^{B}, V̂_{Boot}.

    Y-Achse zeigt nur die Namen der Schätzer, ohne Zusatz.
    Rote Linie = std(rf_pred) für die jeweilige Stufe.
    """

    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ─── Label-Suffix ────────────────────────────────────────────────────────
    if label_style == "plain":
        suf = lambda pw: f" (|{{w_i=0}}|={pw/10:.1f})"
    elif label_style == "math":
        suf = lambda pw: rf"$\frac{{|\{{w_i = 0\}}|}}{{|\{{w_i\}}|}} = {pw/10:.1f}$"
    else:
        raise ValueError("label_style muss 'plain' oder 'math' sein.")

    base = {
        "wB_bias" : r"$\hat{V}_{IJ}^{wB}$",
        "wB"      : r"$\hat{V}_{IJ-U}^{wB}$",
        "B_bias"  : r"$\hat{V}_{IJ}^{B}$",
        "B"       : r"$\hat{V}_{IJ-U}^{B}$",
        "boot"    : r"$\hat{V}_{Boot}$"
    }

    xlabel_map = {
        "averageS": r"$\hat{\sigma}\left(\hat{\theta}_{\tau}(x_{0})\right)$",
        "lowerS":   r"$\hat{\sigma}\left(\hat{\theta}_{\tau}(x_{low})\right)$",
        "higherS":  r"$\hat{\sigma}\left(\hat{\theta}_{\tau}(x_{high})\right)$"
    }

    frames, std_by_pw = [], {}

    # ─── Daten einsammeln ────────────────────────────────────────────────────
    for res, pw in zip(results_list, prop_weights_list):
        v_wB_bias = res["ijk_jahn_var"].clip(lower=0).pipe(np.sqrt)
        v_wB      = res["ijk_u_jahn_var"].clip(lower=0).pipe(np.sqrt)
        v_B_bias  = res["ijk_wager_var"].clip(lower=0).pipe(np.sqrt)
        v_B       = res["ijk_u_wager_var"].clip(lower=0).pipe(np.sqrt)
        v_boot    = res["boot_var"].clip(lower=0).pipe(np.sqrt)

        std_rf = res["rf_pred"].std(ddof=1)
        std_by_pw[pw] = std_rf

        n = len(v_wB_bias)
        frames.append(pd.DataFrame({
            "value": np.concatenate([v_wB_bias, v_wB, v_B_bias, v_B, v_boot]),
            "row":  [f"{base['wB_bias']} {suf(pw)}"] * n +
                    [f"{base['wB']}      {suf(pw)}"] * n +
                    [f"{base['B_bias']}  {suf(pw)}"] * n +
                    [f"{base['B']}       {suf(pw)}"] * n +
                    [f"{base['boot']}    {suf(pw)}"] * n,
            "pw": [pw] * 5 * n
        }))

    df_all = pd.concat(frames, ignore_index=True)

    # ─── Einzelplots je Stufe ────────────────────────────────────────────────
    sns.set_style("whitegrid")

    for pw in prop_weights_list:
        df_pw = df_all[df_all["pw"] == pw].copy()

        # Original-Labels mit w_i Zusatz
        full_labels = [
            f"{base['wB_bias']} {suf(pw)}",
            f"{base['wB']}      {suf(pw)}",
            f"{base['B_bias']}  {suf(pw)}",
            f"{base['B']}       {suf(pw)}",
            f"{base['boot']}    {suf(pw)}"
        ]
        short_labels = [
            base["wB_bias"],
            base["wB"],
            base["B_bias"],
            base["B"],
            base["boot"]
        ]

        fig, ax = plt.subplots(figsize=(5.8, 3.6))

        sns.stripplot(data=df_pw, x="value", y="row",
                      order=full_labels, jitter=0.25,
                      alpha=0.5, size=1.5, color="teal", ax=ax)

        # Mittelwert-Striche
        for i, r in enumerate(full_labels):
            m = df_pw.loc[df_pw["row"] == r, "value"].mean()
            ax.vlines(m, i - 0.4, i + 0.4, color="black",
                      linestyle="--", linewidth=1)

        # Rote Linie = std(rf_pred)
        std_val = std_by_pw[pw]
        ax.vlines(std_val, -0.4, 4.4, color="red", linewidth=1.2, zorder=10)

        # Achsenlayout
        ax.set_ylabel("")
        current_xlim = xlim2[pw] if isinstance(xlim2, dict) else xlim2
        if current_xlim is not None:
            ax.set_xlim(current_xlim)

        # Nur kurze Schätzerlabels anzeigen
        ax.set_yticks(range(len(short_labels)))
        ax.set_yticklabels(short_labels)

        if patient in xlabel_map:
            ax.set_xlabel(xlabel_map[patient])

        fig.tight_layout()

        # Speichern
        if not os.path.exists("help_figures"):
            os.makedirs("help_figures")
        fig.tight_layout()
        fig.savefig(f"help_figures/dctb_all_variances_{patient}_pw{pw}.jpeg", dpi=300)
        plt.close()



# ------------------------------------------------------------------ #
# 1)  Coverage-Funktion (unverändert, nur als Referenz)              #
# ------------------------------------------------------------------ #
def coverage_prop(ijk_std_estimate, S_dach, S):
    """
    Anteil, in dem das 95-%-Intervall   S_dach ± 1.96·ijk_std_estimate
    den wahren Wert S einschließt.
    """
    u = S_dach - 1.96 * ijk_std_estimate
    o = S_dach + 1.96 * ijk_std_estimate
    prop = np.mean((u <= S) & (o >= S))
    return prop.round(4)


# ------------------------------------------------------------------ #
# 2)  Tabelle nur für wB, B, Boot                                    #
# ------------------------------------------------------------------ #

def coverage_table(results_list, prop_weights_list, S_ref_values, label_style="math"):

    if not (len(results_list) == len(prop_weights_list) == len(S_ref_values)):
        raise ValueError("Alle drei Listen müssen gleich lang sein.")

    if label_style == "plain":
        col_wB, col_B, col_boot = "V_IJ-U^wB", "V_IJ-U^B", "V_Boot"
    else:
        col_wB   = r"$\hat{V}_{IJ-U}^{wB}$"
        col_B    = r"$\hat{V}_{IJ-U}^{B}$"
        col_boot = r"$\hat{V}_{Boot}$"

    index_label = r"$p_{w_0}$"
    rows = []

    for res, pw, S_ref in zip(results_list, prop_weights_list, S_ref_values):

        v_wB   = np.sqrt(res["ijk_u_jahn_var"].clip(lower=0))
        v_B    = np.sqrt(res["ijk_u_wager_var"].clip(lower=0))
        v_boot = np.sqrt(res["boot_var"].clip(lower=0))

        S_hat = res["rf_pred"]

        rows.append({
            index_label: pw / 10,
            col_wB  : coverage_prop(v_wB,   S_hat, S_ref),
            col_B   : coverage_prop(v_B,    S_hat, S_ref),
            col_boot: coverage_prop(v_boot, S_hat, S_ref)
        })

    table = pd.DataFrame(rows).set_index(index_label).sort_index()
    print(table.to_latex(escape=False, float_format="%.2f"))

    return table

<>:17: SyntaxWarning: invalid escape sequence '\{'
<>:17: SyntaxWarning: invalid escape sequence '\{'
C:\Users\rehan\AppData\Local\Temp\ipykernel_34504\3692432594.py:17: SyntaxWarning: invalid escape sequence '\{'
  "math"  →  DCTB  $|\{w_i=0\}|=0.1$    ← LaTeX-Look


In [2]:
import os

def _parse_xlim_env(name, default):
    raw = os.environ.get(name)
    if not raw:
        return default
    a, b = raw.split(',')
    return (float(a), float(b))

#exp_save_path = os.environ.get('EXP_SAVE_PATH', r'C:\Users\rehan\OneDrive\00_TO-DO\paper_review\Masterarbeit_Butt\Paper_abgabe\simulation_study\results_in_pdf\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates\121.90058')

exp_save_path = os.environ.get('EXP_SAVE_PATH', r'C:\Users\rehan\OneDrive\00_TO-DO\paper_review\Masterarbeit_Butt\Paper_abgabe\simulation_study\results\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates_events40\121.901767')


patient = os.environ.get('PATIENT_LABEL', 'averageS')

strip_xlim = _parse_xlim_env('STRIP_XLIM', None)
rb_xlim = _parse_xlim_env('RB_XLIM', (-60.0, 100.0))
var_xlim_by_pw = {
    1: _parse_xlim_env('VAR_XLIM_1', None),
    3: _parse_xlim_env('VAR_XLIM_3', None),
    5: _parse_xlim_env('VAR_XLIM_5', None),
}

x = plot_var_bias_without_u2(exp_save_path, rb_xlim[0], rb_xlim[1], patient)
plt.gcf().set_size_inches(8.5, 5.0)
plt.savefig(f"help_figures/MRB_{patient}.jpeg", dpi=300)
plt.close()


In [3]:
#exp_save_path = r'C:\Users\rehan\OneDrive\00_TO-DO\paper_review\Masterarbeit_Butt\Paper_abgabe\simulation_study\results\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates_biasv2\1'
#patient = 'averageS'

#exp_save_path = r'C:\Users\rehan\meine_Repos\Masterarbeit\Paper\results\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates_higherS(tau)\109.90058'
#patient = 'higherS'

#exp_save_path = r'C:\Users\rehan\meine_Repos\Masterarbeit\Paper\results\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates_lowS(tau)\131.90058'
#patient = 'lowerS'


#### Plots


In [4]:
# erstelle verzeichnis, falls nicht vorhanden /help_figures
if not os.path.exists('help_figures'):
    os.makedirs('help_figures')

with open(exp_save_path + '/setting.json') as f:
    exp_settings = json.load(f)
S_t = exp_settings["true_survival_probability[1,3,5]"]

# lade results
results1 = pd.read_csv(exp_save_path + f"/results1__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][0]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][0]}.csv")
results3 = pd.read_csv(exp_save_path + f"/results3__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][1]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][1]}.csv")
results5 = pd.read_csv(exp_save_path + f"/results5__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][2]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][2]}.csv")
x_pred = exp_settings['X_pred_point']

In [5]:
# Ergebnisse-Listen
results_list      = [results1, results3, results5]
prop_weights_list = [1, 3, 5]                 #  → 0.1, 0.3, 0.5

cov_df = coverage_table(results_list,
                        prop_weights_list,
                        S_t,
                        label_style="math")

# Speichere die Coverage-Tabelle als TXT-Datei
cov_df.to_csv('help_figures/coverage_table.txt', sep='\t', index=True)

\begin{tabular}{lrrr}
\toprule
 & $\hat{V}_{IJ-U}^{wB}$ & $\hat{V}_{IJ-U}^{B}$ & $\hat{V}_{Boot}$ \\
$p_{w_0}$ &  &  &  \\
\midrule
0.100000 & 0.20 & 0.10 & 0.80 \\
0.300000 & 0.40 & 0.20 & 0.90 \\
0.500000 & 0.60 & 0.40 & 0.90 \\
\bottomrule
\end{tabular}



In [6]:
print(cov_df)

           $\hat{V}_{IJ-U}^{wB}$  $\hat{V}_{IJ-U}^{B}$  $\hat{V}_{Boot}$
$p_{w_0}$                                                               
0.1                          0.2                   0.1               0.8
0.3                          0.4                   0.2               0.9
0.5                          0.6                   0.4               0.9


In [7]:
plot_dctb(
    results_list      = [results1, results3, results5],
    prop_weights_list = [1, 3, 5],
    S_t               = S_t,     # z. B. np.array([0.25, 0.40, 0.60])
    xlim              = strip_xlim,   # z. B. (0, 1)
    label_style       = "math",   # oder "plain"
    patient         = patient   # "averageS", "lowerS", "higherS"
    )

plot_dctb_variances_separate(
    results_list      = [results1, results3, results5],
    prop_weights_list = [1, 3, 5],
    xlim2             = var_xlim_by_pw,
    patient          = patient,   # "averageS", "lowerS", "higherS"
    label_style       = "math"     # oder "plain"
)

plot_dctb_all_variances_separate(
    results_list      = [results1, results3, results5],
    prop_weights_list = [1, 3, 5],
    xlim2             = var_xlim_by_pw,
    patient          = patient,   # "averageS", "lowerS", "higherS"
    label_style       = "math"     # oder "plain"
)

0.80367821798426
0.7574124562251441
0.795621685202431
[0.8087479549030533, 0.7903560960572573, 0.7572003992560927]
[0.8087479549030533, 0.7903560960572573, 0.7572003992560927]
[0.8087479549030533, 0.7903560960572573, 0.7572003992560927]
ensoring scenario 1 has emp. std.  0.09892519532884628 .
Cencorsing scenario 1 has max predicted std for weighted infinitesimal jackknife  0.05657565456236897 .
Cencorsing scenario 1 has max predicted std for unweighted infinitesimal jackknife  0.0534433315247113 .
Cencorsing scenario 1 has max predicted std for bootstrap  0.13056332658033343 .
Cencorsing scenario 1 has min predicted std for weighted infinitesimal jackknife  0.0 .
Cencorsing scenario 1 has min predicted std for unweighted infinitesimal jackknife  0.0 .
Cencorsing scenario 1 has min predicted std for bootstrap  0.03894878344048502 .
Cencorsing scenario 1 has mean predicted std for weighted infinitesimal jackknife  0.01123766442310623 .
Cencorsing scenario 1 has mean predicted std for unw